# 向日葵地球昼夜延时视频
CPU 即可。原始图片只在云端内存，Drive 保留日分段、缺测记录和成片。日期结束不包含。
先运行准备、配置及规模预估；基准单元格可选，建议先测12帧再做长任务。
来源 NICT / JMA；使用影像请遵守 [NICT 条款](https://himawari8.nict.go.jp/ja/himawari8-about.htm)。
此表单的下载位置是 Google Drive 云端路径；电脑保存目录由浏览器控制，精确目录与续传请用仓库命令行。


In [ ]:
# @title 1. 准备环境
import sys, subprocess, shutil
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'git+https://github.com/cher9lie/himawari-earth-timelapse.git@v0.1.0'])
if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    subprocess.check_call(['apt-get','-qq','update'])
    subprocess.check_call(['apt-get','-qq','install','-y','ffmpeg'])


In [ ]:
# @title 2. 填写配置
START = '2026-07-01' # @param {type:'date'}
END_EXCLUSIVE = '2026-08-01' # @param {type:'date'}
INTERVAL_MINUTES = 10 # @param {type:'integer'}
FPS = 15 # @param {type:'integer'}
WIDTH = 1920 # @param {type:'integer'}
HEIGHT = 1080 # @param {type:'integer'}
UTC_OFFSET = 8.0 # @param {type:'number'}
SAVE_ROOT = '/content/drive/MyDrive/EarthTimelapse' # @param {type:'string'}
SCRATCH = '/content/earth_video_work' # @param {type:'string'}
CRF = 18 # @param {type:'integer'}
PRESET = 'medium' # @param ['ultrafast','veryfast','fast','medium','slow']
from himawari_timelapse.config import Config, estimate
from himawari_timelapse.job import run, benchmark, validate
import json, pathlib
config=Config(start=START,end=END_EXCLUSIVE,interval_minutes=INTERVAL_MINUTES,fps=FPS,
    width=WIDTH,height=HEIGHT,timezone_offset=UTC_OFFSET,save_root=SAVE_ROOT,scratch=SCRATCH,
    workspace='/content/earth_benchmark',crf=CRF,preset=PRESET).validate()
print(json.dumps(estimate(config),ensure_ascii=False,indent=2))


In [ ]:
# @title 3. 可选：测量12帧真实速度
calibration=benchmark(config)
print(json.dumps(estimate(config,calibration),ensure_ascii=False,indent=2))


In [ ]:
# @title 4. 授权 Drive 并制作（相同参数可按日恢复）
from google.colab import drive
drive.mount('/content/drive')
FINAL_VIDEO=run(config)
print('成片：',FINAL_VIDEO)
print(json.dumps(validate(FINAL_VIDEO,config,config.frames),indent=2))


## 取回与结束
从 Drive 网页下载成片，并保留同目录 manifest.json 与 data_gaps.jsonl。
大文件或需要指定本机目录时使用 `earth-video retrieve`，它提供8MiB分块校验与续传。
只有确认已取回成片后，在 Colab 菜单“运行时 → 断开连接并删除运行时”。
运行中连接丢失时先看实际云端进度，不要启动第二份制作或重新挂载。
